# IOC Miner — SecBERT Fine-tuning

Fine-tunes `jackaduma/SecBERT` on CyNER cybersecurity NER dataset.

**Runtime:** GPU T4 (~10-15 min) | GPU A100 (~4-5 min)

**Before running:** `Runtime → Change runtime type → T4 GPU`

**Output:** Fine-tuned model saved to Google Drive at `MyDrive/ioc-miner/secbert-ioc-ner/`

## 1. Setup

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — change runtime to GPU!')

In [ ]:
%%capture
!pip install transformers datasets accelerate seqeval tokenizers

In [ ]:
# Clone repo
!git clone https://github.com/calvinkatoroy/ioc-miner.git
%cd ioc-miner

In [ ]:
# Mount Drive to persist the trained model
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/ioc-miner/secbert-ioc-ner'
print('Model will be saved to:', OUTPUT_DIR)

## 2. Prepare Dataset

In [ ]:
!python training/prepare_dataset.py --output-dir data/cyner_ner

In [ ]:
# Sanity check
from datasets import load_from_disk
import json

ds = load_from_disk('data/cyner_ner')
labels = json.loads(open('data/cyner_ner/label_names.json').read())

print('Label names:', labels)
print(f'Train: {len(ds["train"])} | Val: {len(ds["validation"])} | Test: {len(ds["test"])}')
print('\nSample:', ds['train'][0])

## 3. Fine-tune SecBERT

In [ ]:
import json
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from seqeval.metrics import f1_score, precision_score, recall_score

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME   = 'jackaduma/SecBERT'
DATA_DIR     = 'data/cyner_ner'
EPOCHS       = 5
BATCH_SIZE   = 32   # T4 can handle 32 comfortably
LR           = 2e-5

# ── Load ──────────────────────────────────────────────────────────────────────
dataset     = load_from_disk(DATA_DIR)
label_names = json.loads(open(f'{DATA_DIR}/label_names.json').read())
id2label    = {i: l for i, l in enumerate(label_names)}
label2id    = {l: i for i, l in enumerate(label_names)}
num_labels  = len(label_names)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        t, p = [], []
        for pred, label in zip(pred_seq, label_seq):
            if label == -100:
                continue
            t.append(label_names[label])
            p.append(label_names[pred])
        true_labels.append(t)
        pred_labels.append(p)
    return {
        'precision': precision_score(true_labels, pred_labels),
        'recall':    recall_score(true_labels, pred_labels),
        'f1':        f1_score(true_labels, pred_labels),
    }

# ── Training args ─────────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=True,       # T4 supports FP16
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f'Training on {len(dataset["train"])} examples, {EPOCHS} epochs, batch={BATCH_SIZE}')
print(f'Steps per epoch: {len(dataset["train"]) // BATCH_SIZE}')

In [ ]:
trainer.train()

## 4. Evaluate + Save

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(dataset['test'])

print('\n' + '='*50)
print(f'  F1:        {test_results["eval_f1"]:.4f}')
print(f'  Precision: {test_results["eval_precision"]:.4f}')
print(f'  Recall:    {test_results["eval_recall"]:.4f}')
print('='*50)

In [ ]:
# Save model to Drive
import os, json
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
json.dump(label_names, open(f'{OUTPUT_DIR}/label_names.json', 'w'), indent=2)
json.dump(test_results, open(f'{OUTPUT_DIR}/eval_results.json', 'w'), indent=2)

print('Saved to:', OUTPUT_DIR)
print('Files:', os.listdir(OUTPUT_DIR))

## 5. Smoke Test

In [ ]:
from transformers import pipeline

ner = pipeline(
    'token-classification',
    model=OUTPUT_DIR,
    aggregation_strategy='simple',
)

test_sentences = [
    'The C2 server at 185.220.101.47 received Cobalt Strike beacon callbacks.',
    'CVE-2021-44228 was exploited to deploy a dropper from evil-domain.ru.',
    'The actor used 8.8.8.8 as a DNS resolver to evade detection.',
    'SHA256 hash of the payload: ' + 'a' * 64,
]

for sentence in test_sentences:
    preds = ner(sentence)
    print(f'\nInput: {sentence[:80]}...' if len(sentence) > 80 else f'\nInput: {sentence}')
    for p in preds:
        print(f'  {p["entity_group"]:12} {p["word"]:40} score={p["score"]:.2f}')

## 6. Download Model (optional)

If you want the model locally instead of Drive:

In [ ]:
# Zip and download the model
!zip -r /content/secbert-ioc-ner.zip {OUTPUT_DIR}

from google.colab import files
files.download('/content/secbert-ioc-ner.zip')